# Part 3 -- sweep all 6 datasets overnight

Runs the same quantilizer analyses as part 2 on every dataset uploaded by part 1:

```
mnist                fashionmnist            cifar10
mnist_normalized     fashionmnist_normalized cifar10_normalized
```

For each dataset:
1. Download the `.pt` dump from the HuggingFace repo.
2. Run consultancy + perfect curves, 4-turn + 2-turn matrixplots (judge / claim / winrate),
   4-turn + 2-turn diagonals.
3. Save every matrix as `.npy` and every plot as `.png` (silent, no inline display).
4. Build **3 final combined plots** per dataset:
   * `combined_claim`  -- consultancy, perfect, 2_turn (claim), 4_turn (claim) -- *A's selection accuracy*
   * `combined_judge`  -- consultancy, perfect, judge_2t, judge_4t          -- *Judge accuracy*
   * `combined_all6`   -- all 6 curves on one axes
5. Delete the local dump file (Kaggle working dir is capped at 20 GB; each dump is ~12 GB).
6. Free CUDA memory and continue.

All outputs land in `/kaggle/working/quantilizer_plots/` and
`/kaggle/working/quantilizer_data/`.


In [ ]:
import os, gc, shutil, math, time, itertools
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

CONFIG = {
    # HF dump source (set by part 1)
    'hf_repo_id':       'eruzak/pixeldebatev1-dump',
    'hf_token':         os.environ.get("HF_TOKEN"),
    # Datasets to sweep
    'datasets': [
        'mnist',
        'mnist_normalized',
        'fashionmnist',
        'fashionmnist_normalized',
        'cifar10',
        'cifar10_normalized',
    ],
    # Local paths
    'cache_dir':        '/kaggle/working/dumps_cache',
    'plot_dir':         '/kaggle/working/quantilizer_plots',
    'data_dir':         '/kaggle/working/quantilizer_data',
    # Compute
    'device':           'cuda' if torch.cuda.is_available() else 'cpu',
    'chunk_size':       200,
    'chunk_heatmap':    20,
    'chunk_2turn':      15,
}

for d in (CONFIG['cache_dir'], CONFIG['plot_dir'], CONFIG['data_dir']):
    os.makedirs(d, exist_ok=True)

print(f"device: {CONFIG['device']}")
if CONFIG['device'] == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"free mem: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"datasets to process: {CONFIG['datasets']}")


In [ ]:
def _resolve_hf_token(explicit=None):
    if explicit:
        return explicit
    try:
        from kaggle_secrets import UserSecretsClient
        try:
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    except ImportError:
        pass
    return os.environ.get("HF_TOKEN")


def download_dump(dataset_tag, config=CONFIG):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(
        repo_id=config['hf_repo_id'],
        filename=f"{dataset_tag}.pt",
        repo_type='dataset',
        local_dir=config['cache_dir'],
        token=_resolve_hf_token(config.get('hf_token')),
    )


def load_dump(dataset_tag, config=CONFIG):
    path = download_dump(dataset_tag, config)
    print(f"  loaded {path} ({os.path.getsize(path)/1e9:.2f} GB)")
    dump = torch.load(path, map_location='cpu', weights_only=False)
    print(f"  n={dump['n']} dataset={dump['dataset']} normalized={dump['normalized']}")
    return dump, path


def cleanup_local_cache(config=CONFIG):
    """Wipe the local HF cache to keep Kaggle working dir below quota."""
    cache = config['cache_dir']
    if os.path.exists(cache):
        shutil.rmtree(cache, ignore_errors=True)
    os.makedirs(cache, exist_ok=True)


In [ ]:
def build_index_tables(grid_size=6, squares_to_keep=4, device='cpu'):
    total = grid_size * grid_size
    K = squares_to_keep
    combos = {k: list(itertools.combinations(range(total), k)) for k in (1, 2, 3, K)}
    maps   = {k: {c: i for i, c in enumerate(combos[k])} for k in (2, 3, K)}

    def _children(s_tup, level_up):
        return [maps[level_up][tuple(sorted(s_tup + (x,)))]
                for x in range(total) if x not in s_tup]

    idx_3_to_4 = torch.tensor([_children(s, K) for s in combos[3]], dtype=torch.long)
    idx_2_to_3 = torch.tensor([_children(s, 3) for s in combos[2]], dtype=torch.long)
    idx_1_to_2 = torch.tensor([_children(s, 2) for s in combos[1]], dtype=torch.long)

    n_b_pairs = (total - 2) * (total - 3) // 2
    idx_2_to_4 = torch.zeros((len(combos[2]), n_b_pairs), dtype=torch.long)
    for i, s2 in enumerate(combos[2]):
        rem = [x for x in range(total) if x not in s2]
        for col, b_choices in enumerate(itertools.combinations(rem, 2)):
            idx_2_to_4[i, col] = maps[K][tuple(sorted(s2 + b_choices))]

    return {
        'idx_3_to_4': idx_3_to_4.to(device),
        'idx_2_to_3': idx_2_to_3.to(device),
        'idx_1_to_2': idx_1_to_2.to(device),
        'idx_2_to_4': idx_2_to_4.to(device),
    }


print("Building index tables (once)...")
IDX_TABLES = build_index_tables(grid_size=6, squares_to_keep=4, device=CONFIG['device'])
for k, v in IDX_TABLES.items():
    print(f"  {k}: shape {tuple(v.shape)}")


In [ ]:
def save_array(arr, name, data_dir=None):
    data_dir = data_dir or CONFIG['data_dir']
    path = os.path.join(data_dir, f"{name}.npy")
    a = arr.numpy() if isinstance(arr, torch.Tensor) else np.asarray(arr)
    np.save(path, a)
    return path


def plot_curve(curve, dataset_tag, method,
               x_values=None, q_max=None,
               x_label='', x_log=True, invert_x=True,
               save_dir=None, show=False):
    save_dir = save_dir or CONFIG['plot_dir']
    if isinstance(curve, torch.Tensor):
        curve = curve.numpy()
    n = curve.size
    if x_values is None:
        x_values = np.arange(1, n + 1)
    elif isinstance(x_values, torch.Tensor):
        x_values = x_values.numpy()
    else:
        x_values = np.asarray(x_values)
    if q_max is None:
        q_max = float(x_values.max())
    x = x_values / q_max

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, curve, linewidth=1.5, color='C0')
    if x_log:
        ax.set_xscale('log')
    if invert_x:
        ax.invert_xaxis()
    ax.set_xlabel(x_label or
        r'quantile probability $q/q_{\max}$  '
        r'(decreasing $\rightarrow$ increased capability)')
    ax.set_ylabel('Average judge accuracy')
    ax.set_title(f'Quantilizer {method} -- {dataset_tag}')
    ax.set_ylim(0, 1.02)
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    path = os.path.join(save_dir, f'quantilizer_{method}_{dataset_tag}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show() if show else plt.close(fig)
    return path


def plot_heatmap(matrix, title, save_path,
                 x_values=None, y_values=None,
                 vmin=0.0, vmax=1.0, fontsize=5, show=False):
    arr = matrix.numpy() if isinstance(matrix, torch.Tensor) else np.asarray(matrix)
    QA, QB = arr.shape
    x_values = (np.arange(1, QB + 1) if x_values is None
                else (x_values.numpy() if isinstance(x_values, torch.Tensor)
                      else np.asarray(x_values)))
    y_values = (np.arange(1, QA + 1) if y_values is None
                else (y_values.numpy() if isinstance(y_values, torch.Tensor)
                      else np.asarray(y_values)))

    fig, ax = plt.subplots(figsize=(11, 10))
    im = ax.imshow(arr, cmap='viridis', vmin=vmin, vmax=vmax,
                   aspect='equal', origin='upper')
    ax.invert_xaxis()

    def _ticks(size, vals):
        if size <= 8:
            pos = list(range(size))
        else:
            pos = list(range(0, size, max(1, (size - 1) // 7)))
            if pos[-1] != size - 1:
                pos.append(size - 1)
        return pos, [str(int(vals[p])) for p in pos]

    xt, xtl = _ticks(QB, x_values)
    yt, ytl = _ticks(QA, y_values)
    ax.set_xticks(xt); ax.set_xticklabels(xtl)
    ax.set_yticks(yt); ax.set_yticklabels(ytl)
    ax.set_xlabel(r'$q_B$ (right = $q_B=1$ = minimax)')
    ax.set_ylabel(r'$q_A$ (top = $q_A=1$ = minimax)')
    ax.set_title(title)
    for i in range(QA):
        for j in range(QB):
            v = arr[i, j] * 100
            ax.text(j, i, f"{int(round(v))}", ha='center', va='center',
                    fontsize=fontsize, color='white' if v < 50 else 'black')
    plt.colorbar(im, ax=ax, label='value')
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show() if show else plt.close(fig)
    return save_path


In [ ]:
# Consultancy / perfect quantilizer curves.
#
# Two metrics on top of the same sort:
#   * quantilizer_curve(...)            : judge accuracy in the top-q
#       Cons(q)_judge  = avg over top-q of 1[argmax_c p[m,c] == y]
#       Perf(q)_judge  = same, sort by p[m, gold_pred]
#
#   * quantilizer_selection_curve(...)  : consultant's CLAIM accuracy in the top-q
#       The consultant first picks c*(q) = argmax_c E_{m ~ top-q}[p[m, c]]
#       (a digit choice without quantilizing -- i.e. argmax over an aggregate),
#       then shows one of the top-q masks (the "quantilizing the evidence" step).
#       Sel(q)_cons    = avg over images of 1[c*(q) == y]
#       At q=1 the aggregate is just the top mask -> c*(1) = its argmax,
#         so Sel(1)_cons ~= standard consultancy claim accuracy.
#       At q=N the aggregate is the uniform ensemble -> c*(N) = ensemble argmax.
#
# For "perfect", the consultant's digit choice is the gold classifier's
# prediction (independent of q), so Sel_perfect == mean(gold_pred == y) is a
# scalar; we never compute a curve for it.

def quantilizer_curve(probs, true_labels, sort_key='maxP',
                      gold_pred=None,
                      device=None, chunk_size=None):
    """Judge accuracy curve: avg of 1[argmax_c p[m,c] == y] over top-q (sort_key desc)."""
    device = device or CONFIG['device']
    chunk_size = chunk_size or CONFIG['chunk_size']
    n_images, n_masks, _ = probs.shape
    sum_acc = torch.zeros(n_masks, device=device, dtype=torch.float64)
    q_range = torch.arange(1, n_masks + 1, device=device, dtype=torch.float32)
    t0 = time.time()
    for s in range(0, n_images, chunk_size):
        e = min(s + chunk_size, n_images)
        p = probs[s:e].to(device).float()
        y = true_labels[s:e].to(device).long()
        judge_pred = p.argmax(dim=2)
        correct = (judge_pred == y.unsqueeze(1)).float()
        if sort_key == 'maxP':
            key = p.max(dim=2).values
        elif sort_key == 'gold_pred':
            assert gold_pred is not None
            gp = gold_pred[s:e].to(device).long()
            key = p.gather(2, gp.view(-1, 1, 1).expand(-1, n_masks, 1)).squeeze(2)
        else:
            raise ValueError(sort_key)
        sort_idx = key.argsort(dim=1, descending=True)
        sorted_correct = correct.gather(1, sort_idx)
        sum_acc += (sorted_correct.cumsum(dim=1) / q_range.unsqueeze(0)).sum(dim=0).double()
        del p, judge_pred, correct, key, sort_idx, sorted_correct
        if device == 'cuda':
            torch.cuda.empty_cache()
        print(f"    judge curve chunk {e}/{n_images} | {time.time()-t0:.1f}s", end='\r')
    print()
    return (sum_acc / n_images).float().cpu()


def quantilizer_selection_curve(probs, true_labels, sort_key='maxP',
                                gold_pred=None,
                                device=None, chunk_size=None):
    """Selection / claim accuracy curve.
    For each q: c*(q) = argmax_c (1/q) sum_{m in top-q by sort_key} p[m, c].
    Returns avg over images of 1[c*(q) == y], shape (N_masks,).
    """
    device = device or CONFIG['device']
    chunk_size = chunk_size or CONFIG['chunk_size']
    n_images, n_masks, C = probs.shape
    sum_sel = torch.zeros(n_masks, device=device, dtype=torch.float64)
    t0 = time.time()
    for s in range(0, n_images, chunk_size):
        e = min(s + chunk_size, n_images)
        p = probs[s:e].to(device).float()                           # (b, N, C)
        y = true_labels[s:e].to(device).long()
        if sort_key == 'maxP':
            key = p.max(dim=2).values
        elif sort_key == 'gold_pred':
            assert gold_pred is not None
            gp = gold_pred[s:e].to(device).long()
            key = p.gather(2, gp.view(-1, 1, 1).expand(-1, n_masks, 1)).squeeze(2)
        else:
            raise ValueError(sort_key)
        sort_idx = key.argsort(dim=1, descending=True)
        # reorder p along mask axis by sort_idx
        sort_idx_e = sort_idx.unsqueeze(-1).expand(-1, -1, C)
        p_sorted = p.gather(1, sort_idx_e)                          # (b, N, C)
        cumsum_p = p_sorted.cumsum(dim=1)                           # (b, N, C)
        # argmax over c is invariant to division by q, so skip /q
        c_star = cumsum_p.argmax(dim=2)                             # (b, N)
        sum_sel += (c_star == y.unsqueeze(1)).float().sum(dim=0).double()
        del p, key, sort_idx, sort_idx_e, p_sorted, cumsum_p, c_star
        if device == 'cuda':
            torch.cuda.empty_cache()
        print(f"    sel curve chunk {e}/{n_images} | {time.time()-t0:.1f}s", end='\r')
    print()
    return (sum_sel / n_images).float().cpu()


In [ ]:
def quantilizer_4turn(probs, true_labels, idx_tables,
                     qA_max=36, qB_max=36,
                     chunk=None, device=None, verbose=True):
    chunk = chunk or CONFIG['chunk_heatmap']
    device = device or CONFIG['device']
    idx_3_to_4 = idx_tables['idx_3_to_4']
    idx_2_to_3 = idx_tables['idx_2_to_3']
    idx_1_to_2 = idx_tables['idx_1_to_2']
    QA, QB = qA_max, qB_max
    n, N, C = probs.shape
    judge_sum   = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    claim_sum   = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    winrate_sum = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    qA_dim = torch.arange(QA, device=device)
    qB_dim = torch.arange(QB, device=device)
    qB_t4 = torch.clamp(qB_dim, max=32)
    qA_t3 = torch.clamp(qA_dim, max=33)
    qB_t2 = torch.clamp(qB_dim, max=34)
    qA_t1 = torch.clamp(qA_dim, max=35)
    div33 = torch.arange(1, 34, device=device, dtype=torch.float32)
    div34 = torch.arange(1, 35, device=device, dtype=torch.float32)
    div35 = torch.arange(1, 36, device=device, dtype=torch.float32)
    div36 = torch.arange(1, 37, device=device, dtype=torch.float32)
    t0 = time.time()
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        b = e - s
        p   = probs[s:e].to(device).float()
        y   = true_labels[s:e].to(device).long()
        ja  = (p.argmax(dim=2) == y.unsqueeze(1)).float()

        p_child  = p[:,  idx_3_to_4]
        ja_child = ja[:, idx_3_to_4]
        si = p_child.argsort(dim=2, descending=False)
        cumV = p_child.gather(2, si).cumsum(dim=2) / div33.view(1, 1, -1, 1)
        cumW = ja_child.unsqueeze(-1).expand(-1, -1, -1, C).gather(2, si).cumsum(dim=2) / div33.view(1, 1, -1, 1)
        idx4 = qB_t4.view(1, 1, -1, 1).expand(b, 7140, QB, C)
        V3_c = cumV.gather(2, idx4); W3_c = cumW.gather(2, idx4)
        del p_child, ja_child, si, cumV, cumW, idx4

        V3_cc = V3_c[:, idx_2_to_3]; W3_cc = W3_c[:, idx_2_to_3]
        si = V3_cc.argsort(dim=2, descending=True)
        cumV = V3_cc.gather(2, si).cumsum(dim=2) / div34.view(1, 1, -1, 1, 1)
        cumW = W3_cc.gather(2, si).cumsum(dim=2) / div34.view(1, 1, -1, 1, 1)
        idx3 = qA_t3.view(1, 1, -1, 1, 1).expand(b, 630, QA, QB, C)
        V2_c = cumV.gather(2, idx3); W2_c = cumW.gather(2, idx3)
        del V3_c, W3_c, V3_cc, W3_cc, si, cumV, cumW, idx3

        V2_cc = V2_c[:, idx_1_to_2]; W2_cc = W2_c[:, idx_1_to_2]
        si = V2_cc.argsort(dim=2, descending=False)
        cumV = V2_cc.gather(2, si).cumsum(dim=2) / div35.view(1, 1, -1, 1, 1, 1)
        cumW = W2_cc.gather(2, si).cumsum(dim=2) / div35.view(1, 1, -1, 1, 1, 1)
        idx2 = qB_t2.view(1, 1, 1, 1, -1, 1).expand(b, 36, 1, QA, QB, C)
        V1_c = cumV.gather(2, idx2).squeeze(2)
        W1_c = cumW.gather(2, idx2).squeeze(2)
        del V2_c, W2_c, V2_cc, W2_cc, si, cumV, cumW, idx2

        si = V1_c.argsort(dim=1, descending=True)
        cumV = V1_c.gather(1, si).cumsum(dim=1) / div36.view(1, -1, 1, 1, 1)
        cumW = W1_c.gather(1, si).cumsum(dim=1) / div36.view(1, -1, 1, 1, 1)
        V0_c = cumV[:, qA_t1, qA_dim, :, :]
        W0_c = cumW[:, qA_t1, qA_dim, :, :]
        del V1_c, W1_c, si, cumV, cumW

        claim         = V0_c.argmax(dim=3)
        judge_track   = W0_c.gather(3, claim.unsqueeze(3)).squeeze(3)
        claim_correct = (claim == y.view(b, 1, 1)).float()
        winrate       = V0_c.gather(3, claim.unsqueeze(3)).squeeze(3)
        judge_sum   += judge_track.sum(dim=0).double()
        claim_sum   += claim_correct.sum(dim=0).double()
        winrate_sum += winrate.sum(dim=0).double()
        del V0_c, W0_c, claim, judge_track, claim_correct, winrate

        if device == 'cuda':
            torch.cuda.empty_cache()
        if verbose:
            print(f"    4-turn hm chunk {e}/{n} | {time.time()-t0:.1f}s", end='\r')
    if verbose:
        print()
    return ((judge_sum/n).float().cpu(), (claim_sum/n).float().cpu(), (winrate_sum/n).float().cpu())


In [ ]:
def _logspaced_q(q_max, n=36):
    raw = np.unique(np.round(np.logspace(0, np.log10(q_max), n)).astype(int))
    raw = np.clip(raw, 1, q_max)
    if raw[0] != 1: raw = np.concatenate([[1], raw])
    if raw[-1] != q_max: raw = np.concatenate([raw, [q_max]])
    return torch.tensor(np.unique(raw), dtype=torch.long)


def quantilizer_2turn(probs, true_labels, idx_tables,
                     qA_values=None, qB_values=None,
                     n_qA=36, n_qB=36,
                     chunk=None, device=None, verbose=True):
    chunk = chunk or CONFIG['chunk_2turn']
    device = device or CONFIG['device']
    idx_2_to_4 = idx_tables['idx_2_to_4']
    if qA_values is None: qA_values = _logspaced_q(630, n_qA)
    if qB_values is None: qB_values = _logspaced_q(561, n_qB)
    qA_values = qA_values.to(device); qB_values = qB_values.to(device)
    QA, QB = qA_values.numel(), qB_values.numel()
    qA_pos = qA_values - 1; qB_pos = qB_values - 1

    n, N, C = probs.shape
    judge_sum   = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    claim_sum   = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    winrate_sum = torch.zeros(QA, QB, dtype=torch.float64, device=device)
    div_b = torch.arange(1, 562, device=device, dtype=torch.float32)
    div_a = torch.arange(1, 631, device=device, dtype=torch.float32)

    t0 = time.time()
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        b = e - s
        p   = probs[s:e].to(device).float()
        y   = true_labels[s:e].to(device).long()
        ja  = (p.argmax(dim=2) == y.unsqueeze(1)).float()

        p_b  = p[:, idx_2_to_4]
        ja_b = ja[:, idx_2_to_4]
        si = p_b.argsort(dim=2, descending=False)
        cumV = p_b.gather(2, si).cumsum(dim=2) / div_b.view(1, 1, -1, 1)
        cumW = ja_b.unsqueeze(-1).expand(-1, -1, -1, C).gather(2, si).cumsum(dim=2) / div_b.view(1, 1, -1, 1)
        idxB = qB_pos.view(1, 1, -1, 1).expand(b, 630, QB, C)
        V2 = cumV.gather(2, idxB); W2 = cumW.gather(2, idxB)
        del p_b, ja_b, si, cumV, cumW, idxB

        si = V2.argsort(dim=1, descending=True)
        cumV = V2.gather(1, si).cumsum(dim=1) / div_a.view(1, -1, 1, 1)
        cumW = W2.gather(1, si).cumsum(dim=1) / div_a.view(1, -1, 1, 1)
        idxA = qA_pos.view(1, -1, 1, 1).expand(b, QA, QB, C)
        V0 = cumV.gather(1, idxA); W0 = cumW.gather(1, idxA)
        del V2, W2, si, cumV, cumW, idxA

        claim         = V0.argmax(dim=3)
        judge_track   = W0.gather(3, claim.unsqueeze(3)).squeeze(3)
        claim_correct = (claim == y.view(b, 1, 1)).float()
        winrate       = V0.gather(3, claim.unsqueeze(3)).squeeze(3)
        judge_sum   += judge_track.sum(dim=0).double()
        claim_sum   += claim_correct.sum(dim=0).double()
        winrate_sum += winrate.sum(dim=0).double()
        del V0, W0, claim, judge_track, claim_correct, winrate

        if device == 'cuda':
            torch.cuda.empty_cache()
        if verbose:
            print(f"    2-turn hm chunk {e}/{n} | {time.time()-t0:.1f}s", end='\r')
    if verbose:
        print()
    return ((judge_sum/n).float().cpu(), (claim_sum/n).float().cpu(),
            (winrate_sum/n).float().cpu(), qA_values.cpu(), qB_values.cpu())


In [ ]:
def quantilizer_4turn_diagonal(probs, true_labels, idx_tables,
                              Q=36, chunk=None, device=None, verbose=True):
    chunk = chunk or CONFIG['chunk_heatmap']
    device = device or CONFIG['device']
    idx_3_to_4 = idx_tables['idx_3_to_4']
    idx_2_to_3 = idx_tables['idx_2_to_3']
    idx_1_to_2 = idx_tables['idx_1_to_2']
    n, N, C = probs.shape
    judge_sum   = torch.zeros(Q, dtype=torch.float64, device=device)
    claim_sum   = torch.zeros(Q, dtype=torch.float64, device=device)
    winrate_sum = torch.zeros(Q, dtype=torch.float64, device=device)
    div33 = torch.arange(1, 34, device=device, dtype=torch.float32)
    div34 = torch.arange(1, 35, device=device, dtype=torch.float32)
    div35 = torch.arange(1, 36, device=device, dtype=torch.float32)
    div36 = torch.arange(1, 37, device=device, dtype=torch.float32)
    q_T4 = torch.clamp(torch.arange(Q, device=device), max=32)
    q_T3 = torch.clamp(torch.arange(Q, device=device), max=33)
    q_T2 = torch.clamp(torch.arange(Q, device=device), max=34)
    q_T1 = torch.clamp(torch.arange(Q, device=device), max=35)
    t0 = time.time()
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        b = e - s
        p  = probs[s:e].to(device).float()
        y  = true_labels[s:e].to(device).long()
        ja = (p.argmax(dim=2) == y.unsqueeze(1)).float()

        p_child  = p[:,  idx_3_to_4]
        ja_child = ja[:, idx_3_to_4]
        si = p_child.argsort(dim=2, descending=False)
        cumV = p_child.gather(2, si).cumsum(dim=2) / div33.view(1, 1, -1, 1)
        cumW = ja_child.unsqueeze(-1).expand(-1, -1, -1, C).gather(2, si).cumsum(dim=2) / div33.view(1, 1, -1, 1)
        idx4 = q_T4.view(1, 1, -1, 1).expand(b, 7140, Q, C)
        V3 = cumV.gather(2, idx4); W3 = cumW.gather(2, idx4)
        del p_child, ja_child, si, cumV, cumW, idx4

        V3c = V3[:, idx_2_to_3]; W3c = W3[:, idx_2_to_3]
        si = V3c.argsort(dim=2, descending=True)
        cumV = V3c.gather(2, si).cumsum(dim=2) / div34.view(1, 1, -1, 1, 1)
        cumW = W3c.gather(2, si).cumsum(dim=2) / div34.view(1, 1, -1, 1, 1)
        idx3 = q_T3.view(1, 1, -1, 1, 1).expand(b, 630, Q, Q, C)
        V2 = cumV.gather(2, idx3); W2 = cumW.gather(2, idx3)
        q_id = torch.arange(Q, device=device)
        V2 = V2[:, :, q_id, q_id, :]; W2 = W2[:, :, q_id, q_id, :]
        del V3, W3, V3c, W3c, si, cumV, cumW, idx3

        V2c = V2[:, idx_1_to_2]; W2c = W2[:, idx_1_to_2]
        si = V2c.argsort(dim=2, descending=False)
        cumV = V2c.gather(2, si).cumsum(dim=2) / div35.view(1, 1, -1, 1, 1)
        cumW = W2c.gather(2, si).cumsum(dim=2) / div35.view(1, 1, -1, 1, 1)
        idx2 = q_T2.view(1, 1, -1, 1, 1).expand(b, 36, Q, Q, C)
        V1 = cumV.gather(2, idx2); W1 = cumW.gather(2, idx2)
        V1 = V1[:, :, q_id, q_id, :]; W1 = W1[:, :, q_id, q_id, :]
        del V2, W2, V2c, W2c, si, cumV, cumW, idx2

        si = V1.argsort(dim=1, descending=True)
        cumV = V1.gather(1, si).cumsum(dim=1) / div36.view(1, -1, 1, 1)
        cumW = W1.gather(1, si).cumsum(dim=1) / div36.view(1, -1, 1, 1)
        idx1 = q_T1.view(1, -1, 1, 1).expand(b, Q, Q, C)
        V0 = cumV.gather(1, idx1); W0 = cumW.gather(1, idx1)
        V0 = V0[:, q_id, q_id, :]; W0 = W0[:, q_id, q_id, :]
        del V1, W1, si, cumV, cumW, idx1

        claim         = V0.argmax(dim=2)
        judge_track   = W0.gather(2, claim.unsqueeze(2)).squeeze(2)
        claim_correct = (claim == y.view(b, 1)).float()
        winrate       = V0.gather(2, claim.unsqueeze(2)).squeeze(2)
        judge_sum   += judge_track.sum(dim=0).double()
        claim_sum   += claim_correct.sum(dim=0).double()
        winrate_sum += winrate.sum(dim=0).double()
        del V0, W0, claim, judge_track, claim_correct, winrate

        if device == 'cuda':
            torch.cuda.empty_cache()
        if verbose:
            print(f"    4t diag chunk {e}/{n} | {time.time()-t0:.1f}s", end='\r')
    if verbose:
        print()
    return ((judge_sum/n).float().cpu(), (claim_sum/n).float().cpu(),
            (winrate_sum/n).float().cpu())


In [ ]:
def quantilizer_2turn_diagonal(probs, true_labels, idx_tables,
                              q_values=None, chunk=None, device=None, verbose=True):
    chunk = chunk or CONFIG['chunk_2turn']
    device = device or CONFIG['device']
    idx_2_to_4 = idx_tables['idx_2_to_4']
    if q_values is None:
        q_values = _logspaced_q(630, 60)
    q_values = q_values.to(device)
    Q = q_values.numel()
    n, N, C = probs.shape
    judge_sum   = torch.zeros(Q, dtype=torch.float64, device=device)
    claim_sum   = torch.zeros(Q, dtype=torch.float64, device=device)
    winrate_sum = torch.zeros(Q, dtype=torch.float64, device=device)
    qB_idx = torch.clamp(q_values, max=561) - 1
    qA_idx = torch.clamp(q_values, max=630) - 1
    div_b = torch.arange(1, 562, device=device, dtype=torch.float32)
    div_a = torch.arange(1, 631, device=device, dtype=torch.float32)
    t0 = time.time()
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        b = e - s
        p  = probs[s:e].to(device).float()
        y  = true_labels[s:e].to(device).long()
        ja = (p.argmax(dim=2) == y.unsqueeze(1)).float()

        p_b  = p[:, idx_2_to_4]
        ja_b = ja[:, idx_2_to_4]
        si = p_b.argsort(dim=2, descending=False)
        cumV = p_b.gather(2, si).cumsum(dim=2) / div_b.view(1, 1, -1, 1)
        cumW = ja_b.unsqueeze(-1).expand(-1, -1, -1, C).gather(2, si).cumsum(dim=2) / div_b.view(1, 1, -1, 1)
        idxB = qB_idx.view(1, 1, -1, 1).expand(b, 630, Q, C)
        V2 = cumV.gather(2, idxB); W2 = cumW.gather(2, idxB)
        del p_b, ja_b, si, cumV, cumW, idxB

        si = V2.argsort(dim=1, descending=True)
        cumV = V2.gather(1, si).cumsum(dim=1) / div_a.view(1, -1, 1, 1)
        cumW = W2.gather(1, si).cumsum(dim=1) / div_a.view(1, -1, 1, 1)
        idxA = qA_idx.view(1, -1, 1, 1).expand(b, Q, Q, C)
        V0 = cumV.gather(1, idxA); W0 = cumW.gather(1, idxA)
        q_id = torch.arange(Q, device=device)
        V0 = V0[:, q_id, q_id, :]; W0 = W0[:, q_id, q_id, :]
        del V2, W2, si, cumV, cumW, idxA

        claim         = V0.argmax(dim=2)
        judge_track   = W0.gather(2, claim.unsqueeze(2)).squeeze(2)
        claim_correct = (claim == y.view(b, 1)).float()
        winrate       = V0.gather(2, claim.unsqueeze(2)).squeeze(2)
        judge_sum   += judge_track.sum(dim=0).double()
        claim_sum   += claim_correct.sum(dim=0).double()
        winrate_sum += winrate.sum(dim=0).double()
        del V0, W0, claim, judge_track, claim_correct, winrate

        if device == 'cuda':
            torch.cuda.empty_cache()
        if verbose:
            print(f"    2t diag chunk {e}/{n} | {time.time()-t0:.1f}s", end='\r')
    if verbose:
        print()
    return (q_values.cpu(),
            (judge_sum/n).float().cpu(),
            (claim_sum/n).float().cpu(),
            (winrate_sum/n).float().cpu())


In [ ]:
def _combined_plot(curves, title, save_path, hlines=None):
    """`curves` = list of (label, color, x_array, y_array).
    `hlines` = optional list of (label, color, y_value) for horizontal lines.
    All lines only, log x, x-axis inverted (right = high capability)."""
    fig, ax = plt.subplots(figsize=(10, 6))
    if hlines:
        for label, color, yv in hlines:
            ax.axhline(yv, color=color, lw=1.5, linestyle='--',
                       label=f'{label} ({yv*100:.2f}%)')
    for label, color, x, y in curves:
        x = np.asarray(x); y = np.asarray(y)
        ax.plot(x, y, color=color, lw=1.5, label=label)
    ax.set_xscale('log')
    ax.invert_xaxis()
    ax.set_xlabel(r'quantile probability $q/q_{\max}$  '
                  r'(decreasing $\rightarrow$ increased capability)')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.set_ylim(0, 1.02)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='lower right', frameon=True)
    ax.text(0.02, -0.16, 'uniform random / low capability',
            transform=ax.transAxes, ha='left', fontsize=9, color='gray')
    ax.text(0.98, -0.16, 'greedy / minimax / high capability',
            transform=ax.transAxes, ha='right', fontsize=9, color='gray')
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def final_plots(dataset_tag,
                curve_cons_judge,    # quantilizer judge accuracy (sort by maxP)
                curve_cons_sel,      # NEW consultancy selection accuracy curve
                curve_perf_judge,    # quantilizer judge accuracy (sort by gold_pred)
                gold_acc,            # scalar: mean(gold_pred == y) -- = perfect selection acc
                diag_claim_4t, diag_judge_4t,
                q_vals_2t, diag_claim_2t, diag_judge_2t,
                plot_dir=None):
    """Build the 2 final combined plots for one dataset."""
    plot_dir = plot_dir or CONFIG['plot_dir']
    x_cons   = np.arange(1, curve_cons_judge.numel() + 1) / curve_cons_judge.numel()
    x_cons_s = np.arange(1, curve_cons_sel  .numel() + 1) / curve_cons_sel  .numel()
    x_perf   = np.arange(1, curve_perf_judge.numel() + 1) / curve_perf_judge.numel()
    x_4t     = np.arange(1, diag_claim_4t.numel() + 1) / 36
    x_2t     = q_vals_2t.numpy() / 630

    # --- Plot 1: Selection accuracy (A's claim) ---
    p1 = os.path.join(plot_dir, f'combined_selection_{dataset_tag}.png')
    _combined_plot(
        curves=[
            ('consultancy quantilizer (claim)', 'C0', x_cons_s, curve_cons_sel.numpy()),
            ('2_turn  (A claim)',               'C1', x_2t,     diag_claim_2t.numpy()),
            ('4_turn  (A claim)',               'C3', x_4t,     diag_claim_4t.numpy()),
        ],
        hlines=[('perfect / gold-classifier', 'C2', gold_acc)],
        title=f"Selection accuracy (A's claim correct) -- {dataset_tag}",
        save_path=p1,
    )

    # --- Plot 2: Judge accuracy ---
    p2 = os.path.join(plot_dir, f'combined_judge_{dataset_tag}.png')
    _combined_plot(
        curves=[
            ('consultancy quantilizer (judge)', 'C0', x_cons,   curve_cons_judge.numpy()),
            ('perfect (judge)',                 'C2', x_perf,   curve_perf_judge.numpy()),
            ('judge_2t',                        'C1', x_2t,     diag_judge_2t.numpy()),
            ('judge_4t',                        'C3', x_4t,     diag_judge_4t.numpy()),
        ],
        title=f'Judge accuracy -- {dataset_tag}',
        save_path=p2,
    )

    return [p1, p2]


In [ ]:
def _plot_horizontal_line(value, dataset_tag, method, save_dir=None):
    """Save a simple line plot of a single horizontal value (e.g. perfect selection)."""
    save_dir = save_dir or CONFIG['plot_dir']
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.axhline(value, color='C2', lw=2,
               label=f'{method} = {value*100:.2f}%')
    ax.set_xscale('log')
    ax.invert_xaxis()
    ax.set_xlim(1.0, 1.0/58905)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel(r'quantile probability $q/q_{\max}$  '
                  r'(decreasing $\rightarrow$ increased capability)')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Quantilizer {method} -- {dataset_tag}')
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(loc='lower right')
    fig.tight_layout()
    path = os.path.join(save_dir, f'quantilizer_{method}_{dataset_tag}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return path


def process_dataset(tag, config=CONFIG):
    """Full pipeline for one dataset: download, run, save, plot, clean up."""
    t_start = time.time()
    print(f"\n{'='*70}\n>>> {tag}\n{'='*70}")

    # --- 1. Download + load ---
    print(f"[1/8] downloading + loading {tag}.pt ...")
    dump, local_path = load_dump(tag, config)
    probs       = dump['all_probs']
    true_labels = dump['true'].long()
    gold_pred   = dump['gold_pred'].long()
    n_images, n_masks, num_classes = probs.shape
    print(f"      {n_images} images x {n_masks} masks x {num_classes} classes")

    gold_acc = (gold_pred == true_labels).float().mean().item()
    print(f"      gold classifier accuracy (== perfect selection): {gold_acc*100:.2f}%")

    # --- 2. Consultancy quantilizer: judge curve + selection curve ---
    print(f"[2/8] consultancy quantilizer curves (judge + selection) ...")
    curve_cons_judge = quantilizer_curve(probs, true_labels, sort_key='maxP')
    save_array(curve_cons_judge, f'quantilizer_consultancy_judge_{tag}')
    plot_curve(curve_cons_judge, tag, 'consultancy_judge', show=False)

    curve_cons_sel = quantilizer_selection_curve(probs, true_labels, sort_key='maxP')
    save_array(curve_cons_sel,   f'quantilizer_consultancy_selection_{tag}')
    plot_curve(curve_cons_sel,   tag, 'consultancy_selection', show=False)

    # --- 3. Perfect quantilizer judge curve (selection is the constant gold_acc) ---
    print(f"[3/8] perfect quantilizer judge curve ...")
    curve_perf = quantilizer_curve(probs, true_labels, sort_key='gold_pred',
                                   gold_pred=gold_pred)
    save_array(curve_perf, f'quantilizer_perfect_{tag}')
    plot_curve(curve_perf, tag, 'perfect', show=False)

    # Perfect selection = constant gold accuracy. Save scalar + draw a flat line.
    save_array(np.array([gold_acc]), f'perfect_selection_{tag}')
    _plot_horizontal_line(gold_acc, tag, 'perfect_selection')

    # --- 4. 4-turn matrixplots ---
    print(f"[4/8] 4-turn 36x36 heatmaps ...")
    judge4_hm, claim4_hm, winrate4_hm = quantilizer_4turn(
        probs, true_labels, IDX_TABLES, qA_max=36, qB_max=36)
    save_array(judge4_hm,   f'quantilizer_judge_4t_{tag}')
    save_array(claim4_hm,   f'quantilizer_4_turn_{tag}')
    save_array(winrate4_hm, f'quantilizer_winrate_4turn_{tag}')
    q_4t_lin = np.arange(1, 37)
    plot_heatmap(judge4_hm,   f'judge_4t -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_judge_4t_{tag}.png'),
                 x_values=q_4t_lin, y_values=q_4t_lin)
    plot_heatmap(claim4_hm,   f'4_turn (A\'s selection accuracy) -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_4_turn_{tag}.png'),
                 x_values=q_4t_lin, y_values=q_4t_lin)
    plot_heatmap(winrate4_hm, f'4-turn winrate -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_winrate_4turn_{tag}.png'),
                 x_values=q_4t_lin, y_values=q_4t_lin)

    # --- 5. 2-turn matrixplots (log-spaced) ---
    print(f"[5/8] 2-turn log-spaced heatmaps ...")
    judge2_hm, claim2_hm, winrate2_hm, qA_2t_vals, qB_2t_vals = quantilizer_2turn(
        probs, true_labels, IDX_TABLES)
    save_array(judge2_hm,   f'quantilizer_judge_2t_{tag}')
    save_array(claim2_hm,   f'quantilizer_2_turn_{tag}')
    save_array(winrate2_hm, f'quantilizer_winrate_2turn_{tag}')
    save_array(qA_2t_vals.float(), f'quantilizer_qA_2t_{tag}')
    save_array(qB_2t_vals.float(), f'quantilizer_qB_2t_{tag}')
    plot_heatmap(judge2_hm,   f'judge_2t -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_judge_2t_{tag}.png'),
                 x_values=qB_2t_vals, y_values=qA_2t_vals)
    plot_heatmap(claim2_hm,   f'2_turn (A\'s selection accuracy) -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_2_turn_{tag}.png'),
                 x_values=qB_2t_vals, y_values=qA_2t_vals)
    plot_heatmap(winrate2_hm, f'2-turn winrate -- {tag}',
                 os.path.join(config['plot_dir'], f'quantilizer_winrate_2turn_{tag}.png'),
                 x_values=qB_2t_vals, y_values=qA_2t_vals)

    # --- 6. Diagonals (extended range, for combined plots) ---
    print(f"[6/8] 4-turn + 2-turn diagonals ...")
    diag_judge_4t, diag_claim_4t, diag_winrate_4t = quantilizer_4turn_diagonal(
        probs, true_labels, IDX_TABLES, Q=36)
    save_array(diag_judge_4t,   f'quantilizer_diag_judge_4t_{tag}')
    save_array(diag_claim_4t,   f'quantilizer_diag_4_turn_{tag}')
    save_array(diag_winrate_4t, f'quantilizer_diag_winrate_4turn_{tag}')
    plot_curve(diag_judge_4t, tag, 'diag_judge_4t',
               x_values=np.arange(1, 37), q_max=36)

    q_vals_2t, diag_judge_2t, diag_claim_2t, diag_winrate_2t = quantilizer_2turn_diagonal(
        probs, true_labels, IDX_TABLES)
    save_array(q_vals_2t.float(),  f'quantilizer_diag_qvals_2t_{tag}')
    save_array(diag_judge_2t,      f'quantilizer_diag_judge_2t_{tag}')
    save_array(diag_claim_2t,      f'quantilizer_diag_2_turn_{tag}')
    save_array(diag_winrate_2t,    f'quantilizer_diag_winrate_2turn_{tag}')
    plot_curve(diag_judge_2t, tag, 'diag_judge_2t',
               x_values=q_vals_2t.numpy(), q_max=630)

    # --- 7. Final 2 combined plots ---
    print(f"[7/8] final combined plots (selection + judge) ...")
    paths = final_plots(tag,
                        curve_cons_judge, curve_cons_sel,
                        curve_perf, gold_acc,
                        diag_claim_4t, diag_judge_4t,
                        q_vals_2t, diag_claim_2t, diag_judge_2t)
    for p in paths:
        print(f"      saved {p}")

    # --- 8. Cleanup ---
    print(f"[8/8] cleanup ...")
    del probs, true_labels, gold_pred, dump
    del curve_cons_judge, curve_cons_sel, curve_perf
    del judge4_hm, claim4_hm, winrate4_hm
    del judge2_hm, claim2_hm, winrate2_hm, qA_2t_vals, qB_2t_vals
    del diag_judge_4t, diag_claim_4t, diag_winrate_4t
    del q_vals_2t, diag_judge_2t, diag_claim_2t, diag_winrate_2t
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    cleanup_local_cache(config)
    print(f"      total: {(time.time()-t_start)/60:.1f} min  | cache wiped")


In [ ]:
# ============================================================
# Main sweep -- runs all configured datasets sequentially.
# ============================================================
sweep_t0 = time.time()
for tag in CONFIG['datasets']:
    try:
        process_dataset(tag)
    except Exception as ex:
        import traceback
        print(f"\n!!! ERROR on {tag}: {ex}")
        traceback.print_exc()
        # try to clean up and continue with the next dataset
        cleanup_local_cache()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n\n{'='*70}")
print(f"SWEEP COMPLETE -- total wall time {(time.time()-sweep_t0)/60:.1f} min")
print(f"{'='*70}")

# Final inventory
print("\nGenerated files:")
import glob
for p in sorted(glob.glob(os.path.join(CONFIG['plot_dir'], '*.png'))):
    print(f"  plot: {p}")
print()
for p in sorted(glob.glob(os.path.join(CONFIG['data_dir'], '*.npy'))):
    print(f"  data: {p}")
